# Post Pandemic Regime Shifts in Labor Market: Pulling Data from FRED

## Purpose

## Research Context

## Inputs and Outputs

## Imports and Configuration

In [1]:
import re
import time
from pathlib import Path
import requests

import numpy as np
import pandas as pd

from fredapi import Fred

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 160)

In [2]:
repo_root = Path.home() / "Documents" / "Coding" / "ML_LaborTightnessRegimeShift"
# repo_root = Path(r"C:\Users\sumai\Documents\ML_LaborTightnessRegimeShift")

data_root = repo_root / "data"
raw_dir = data_root / "raw"
external_dir = raw_dir / "external_sources"
fred_dir = raw_dir / "fred"

fred_dir.mkdir(parents=True, exist_ok=True)
external_dir.mkdir(parents=True, exist_ok=True)

In [31]:
RAW_API_KEY = """
 Insert Key Here
"""

api_key = re.sub(r"\s+", "", RAW_API_KEY)

fred = Fred(api_key=api_key)

start_date = "2000-01-01"
end_date = pd.Timestamp.today().strftime("%Y-%m-%d")

In [4]:
series_map = {
    # "job_openings_level_sa": "JTSJOLSL",
    "job_openings_level": "JTSJOL",

    "hires_rate": "JTSHIR",
    "hires_level": "JTSHIL",
    "quits_rate": "JTSQUR",
    "quits_level": "JTSQUL",
    "total_separations_rate": "JTSTSR",
    "total_separations_level": "JTSTSL",
    "layoffs_discharges_level": "JTSLDL",
    "layoffs_discharges_rate": "JTSLDR",

    "unemployment_rate": "UNRATE",
    "unemployment_level": "UNEMPLOY",

    "payrolls_total_nonfarm": "PAYEMS",
    "payrolls_total_private": "USPRIV",
    "payrolls_manufacturing": "MANEMP",
    "payrolls_service_providing": "SRVPRD",
    "payrolls_construction": "USCONS",

    "prime_age_lfpr": "LNS11300060",
    "labor_force_participation_rate": "CIVPART",
    "employment_population_ratio": "EMRATIO",
    "u6_unemployment_rate": "U6RATE",
    "prime_age_unemployment_rate": "LNS13025703",

    "average_weeks_unemployed": "UEMPMEAN",

    "avg_hourly_earnings_total_private": "CES0500000003",
    "avg_weekly_earnings_total_private": "CES0500000011",

    "avg_weekly_hours_total_private": "AWHAETP",

    "avg_weekly_earnings_manufacturing": "CES3000000008",
    "avg_hourly_earnings_manufacturing": "CES3000000003",

    "cpi_all_items": "CPIAUCSL",
    "cpi_core": "CPILFESL",
    "cpi_all_items_less_shelter": "CUSR0000SA0L2",
    "cpi_services_less_rent_of_shelter": "CUUR0000SASL2RS",
    "cpi_services_less_energy_services": "CUSR0000SASLE",
    "cpi_shelter": "CUSR0000SEHA",

    "pce_price_index": "PCEPI",
    "pce_core_price_index": "PCEPILFE",

    "personal_saving_rate": "PMSAVE",
    "real_disposable_personal_income": "DSPIC96",
    "real_personal_income": "RPI",
    "real_personal_income_less_transfers": "W875RX1",

    "retail_sales_advance": "RSAFS",
    "real_retail_sales": "RRSFS",

    "industrial_production": "INDPRO",
    "industrial_production_manufacturing": "IPMAN",
    "capacity_utilization": "CUMFNS",

    "housing_starts": "HOUST",
    "building_permits": "PERMIT",
    "new_one_family_houses_sold": "HSN1F",

    "consumer_sentiment": "UMCSENT",

    "durable_goods_orders": "DGORDER",
    "capital_goods_orders_ex_aircraft": "NEWORDER",

    "commercial_industrial_loans": "BUSLOANS",
    "total_bank_credit": "TOTBKCR",
    "m2_money_stock": "M2SL",

    "treasury_1m": "DGS1MO",
    "treasury_3m": "DGS3MO",
    "treasury_2y": "DGS2",
    "treasury_10y": "GS10",
    "fed_funds_rate": "FEDFUNDS",

    "aaa_corporate_yield": "AAA",
    "baa_corporate_yield": "BAA",
    "high_yield_oas": "BAMLH0A0HYM2",
    "bbb_oas": "BAMLC0A4CBBB",

    "breakeven_5y": "T5YIE",
    "forward_5y5y_inflation": "T5YIFR",

    "michigan_inflation_1y": "MICH",
    "michigan_inflation_5y": "MICH5Y",

    "recession_indicator": "USREC",
}

print("Series Count:", len(series_map))
print("Start Date:", start_date)
print("End Date:", end_date)

Series Count: 67
Start Date: 2000-01-01
End Date: 2026-03-19


In [5]:
def request_with_retry(url, session=None, max_tries=5, timeout=120, sleep_seconds=2.0):
    last_error = None

    if session is None:
        session = requests.Session()

    for attempt in range(1, max_tries + 1):
        try:
            response = session.get(url, timeout=timeout)
            response.raise_for_status()
            return response
        except Exception as e:
            last_error = e
            if attempt < max_tries:
                wait_time = sleep_seconds * attempt
                print(f"Retrying Download: attempt {attempt}/{max_tries} | wait {wait_time:.1f}s | {url}")
                time.sleep(wait_time)

    raise RuntimeError(f"Download failed after {max_tries} tries: {url} | {last_error}")

In [6]:
def fred_get_series_retry(series_id, observation_start, observation_end, max_tries=5, sleep_seconds=1.5):
    last_error = None

    for attempt in range(1, max_tries + 1):
        try:
            series = fred.get_series(
                series_id,
                observation_start=observation_start,
                observation_end=observation_end,
            )
            return series
        except Exception as e:
            last_error = e
            msg = str(e).lower()

            if ("too many requests" in msg or "rate limit" in msg) and attempt < max_tries:
                wait_time = sleep_seconds * attempt
                print(f"Retrying FRED Series: {series_id} | attempt {attempt}/{max_tries} | wait {wait_time:.1f}s")
                time.sleep(wait_time)
                continue

            if attempt < max_tries:
                wait_time = sleep_seconds * attempt
                print(f"Retrying FRED Series: {series_id} | attempt {attempt}/{max_tries} | wait {wait_time:.1f}s | {e}")
                time.sleep(wait_time)
            else:
                raise

    raise RuntimeError(f"FRED series failed: {series_id} | {last_error}")

In [7]:
def fred_get_info_retry(series_id, max_tries=5, sleep_seconds=1.5):
    last_error = None

    for attempt in range(1, max_tries + 1):
        try:
            info = fred.get_series_info(series_id)
            return info
        except Exception as e:
            last_error = e
            msg = str(e).lower()

            if ("too many requests" in msg or "rate limit" in msg) and attempt < max_tries:
                wait_time = sleep_seconds * attempt
                print(f"Retrying FRED Metadata: {series_id} | attempt {attempt}/{max_tries} | wait {wait_time:.1f}s")
                time.sleep(wait_time)
                continue

            if attempt < max_tries:
                wait_time = sleep_seconds * attempt
                print(f"Retrying FRED Metadata: {series_id} | attempt {attempt}/{max_tries} | wait {wait_time:.1f}s | {e}")
                time.sleep(wait_time)
            else:
                return pd.Series(dtype="object")

    return pd.Series(dtype="object")

In [8]:
def get_series_with_info(series_id, col_name, start_date=start_date, end_date=end_date):
    series = fred_get_series_retry(
        series_id,
        observation_start=start_date,
        observation_end=end_date,
    )

    series = pd.Series(series, name=col_name)
    series.index = pd.to_datetime(series.index)

    info = fred_get_info_retry(series_id)
    return series, info

In [9]:
def convert_to_month_start(series, freq_code):
    freq_code = str(freq_code).strip().upper()

    if freq_code == "M":
        monthly = series.copy()
    elif freq_code in {"D", "W", "BW"}:
        monthly = series.resample("MS").mean()
    elif freq_code == "Q":
        monthly = series.resample("MS").ffill()
    else:
        monthly = series.resample("MS").mean()

    monthly.index = pd.to_datetime(monthly.index)
    monthly = monthly.sort_index()
    return monthly

In [10]:
def classify_missingness(series, observation_start):
    if pd.isna(observation_start):
        observation_start = pd.Timestamp.min

    observation_start = pd.to_datetime(observation_start, errors="coerce")
    if pd.isna(observation_start):
        observation_start = pd.Timestamp.min

    full_index = pd.Index(series.index)
    pre_start_mask = full_index < observation_start
    post_start_mask = full_index >= observation_start

    structural_missing = int(series.loc[pre_start_mask].isna().sum()) if len(series.loc[pre_start_mask]) > 0 else 0
    active_missing = int(series.loc[post_start_mask].isna().sum()) if len(series.loc[post_start_mask]) > 0 else 0

    return structural_missing, active_missing

In [11]:
series_list = []
meta_rows = []
failed_rows = []

for i, (col_name, series_id) in enumerate(series_map.items(), start=1):
    try:
        print(f"Pulling FRED Series: {i}/{len(series_map)} | {col_name} -> {series_id}")

        series, info = get_series_with_info(series_id, col_name)

        if series.empty:
            failed_rows.append({
                "column_name": col_name,
                "series_id": series_id,
                "error": "empty_series",
            })
            continue

        freq_code = info.get("frequency_short", "")
        monthly = convert_to_month_start(series, freq_code)
        monthly.name = col_name

        series_list.append(monthly)

        obs_start = info.get("observation_start", "")
        obs_end = info.get("observation_end", "")
        structural_missing, active_missing = classify_missingness(monthly, obs_start)

        meta_rows.append({
            "column_name": col_name,
            "series_id": series_id,
            "title": info.get("title", ""),
            "frequency_short": str(freq_code).strip().upper(),
            "units": info.get("units", ""),
            "seasonal_adjustment": info.get("seasonal_adjustment", ""),
            "last_updated": info.get("last_updated", ""),
            "observation_start": obs_start,
            "observation_end": obs_end,
            "notes": info.get("notes", ""),
            "raw_nonmissing_count": int(series.notna().sum()),
            "monthly_nonmissing_count": int(monthly.notna().sum()),
            "monthly_structural_missing_count": structural_missing,
            "monthly_active_missing_count": active_missing,
        })

        time.sleep(0.35)

    except Exception as e:
        failed_rows.append({
            "column_name": col_name,
            "series_id": series_id,
            "error": str(e),
        })

if not series_list:
    raise RuntimeError("No FRED downloads succeeded. Check the API key, internet connection, or series map.")

Pulling FRED Series: 1/67 | job_openings_level -> JTSJOL
Pulling FRED Series: 2/67 | hires_rate -> JTSHIR
Pulling FRED Series: 3/67 | hires_level -> JTSHIL
Pulling FRED Series: 4/67 | quits_rate -> JTSQUR
Pulling FRED Series: 5/67 | quits_level -> JTSQUL
Pulling FRED Series: 6/67 | total_separations_rate -> JTSTSR
Pulling FRED Series: 7/67 | total_separations_level -> JTSTSL
Pulling FRED Series: 8/67 | layoffs_discharges_level -> JTSLDL
Pulling FRED Series: 9/67 | layoffs_discharges_rate -> JTSLDR
Pulling FRED Series: 10/67 | unemployment_rate -> UNRATE
Pulling FRED Series: 11/67 | unemployment_level -> UNEMPLOY
Pulling FRED Series: 12/67 | payrolls_total_nonfarm -> PAYEMS
Pulling FRED Series: 13/67 | payrolls_total_private -> USPRIV
Pulling FRED Series: 14/67 | payrolls_manufacturing -> MANEMP
Pulling FRED Series: 15/67 | payrolls_service_providing -> SRVPRD
Pulling FRED Series: 16/67 | payrolls_construction -> USCONS
Pulling FRED Series: 17/67 | prime_age_lfpr -> LNS11300060
Pulling 

In [13]:
fred_raw = pd.concat(series_list, axis=1, sort=False).sort_index()
fred_raw = fred_raw.loc[
    (fred_raw.index >= pd.Timestamp(start_date)) &
    (fred_raw.index <= pd.Timestamp(end_date))
].copy()

fred_raw.index.name = "date"

fred_meta = pd.DataFrame(meta_rows).sort_values("column_name").reset_index(drop=True)
fred_failed = pd.DataFrame(failed_rows)

fred_missing = (
    fred_raw.isna()
    .sum()
    .sort_values(ascending=False)
    .rename("missing_count")
    .to_frame()
)

fred_missing["missing_pct"] = fred_missing["missing_count"] / len(fred_raw)

if not fred_meta.empty:
    fred_missing = fred_missing.merge(
        fred_meta[["column_name", "observation_start", "observation_end", "monthly_structural_missing_count", "monthly_active_missing_count"]],
        left_index=True,
        right_on="column_name",
        how="left",
    ).set_index("column_name")

In [18]:
fred_csv_path = fred_dir / "fred_raw_monthly.csv"
fred_meta_path = fred_dir / "fred_metadata.csv"
fred_failed_path = fred_dir / "fred_failed_series.csv"
fred_missing_path = fred_dir / "fred_missing_summary.csv"
fred_manifest_path = fred_dir / "fred_output_manifest.csv"

fred_raw.to_csv(fred_csv_path)
fred_meta.to_csv(fred_meta_path, index=False)
fred_failed.to_csv(fred_failed_path, index=False)
fred_missing.to_csv(fred_missing_path)

In [19]:
fred_manifest = pd.DataFrame(
    [
        {
            "table_name": "fred_raw_monthly",
            "path": str(fred_csv_path),
            "rows": len(fred_raw),
            "cols": fred_raw.shape[1],
        },
        {
            "table_name": "fred_metadata",
            "path": str(fred_meta_path),
            "rows": len(fred_meta),
            "cols": fred_meta.shape[1] if not fred_meta.empty else 0,
        },
        {
            "table_name": "fred_failed_series",
            "path": str(fred_failed_path),
            "rows": len(fred_failed),
            "cols": fred_failed.shape[1] if not fred_failed.empty else 0,
        },
        {
            "table_name": "fred_missing_summary",
            "path": str(fred_missing_path),
            "rows": len(fred_missing),
            "cols": fred_missing.shape[1],
        },
    ]
)

fred_manifest.to_csv(fred_manifest_path, index=False)

In [20]:
print("FRED Raw Shape:", fred_raw.shape)
print("FRED Date Range:", fred_raw.index.min(), "to", fred_raw.index.max())
print("CSV:", fred_csv_path)
print("Metadata:", fred_meta_path)
print("Failed Series:", fred_failed_path)
print("Missing Summary:", fred_missing_path)

display(fred_raw.head())
display(fred_meta.head(20))
display(fred_failed.head(20))
display(fred_missing.head(25))

FRED Raw Shape: (315, 66)
FRED Date Range: 2000-01-01 00:00:00 to 2026-03-01 00:00:00
CSV: /Users/Sumaitat/Documents/Coding/ML_LaborTightnessRegimeShift/data/raw/fred/fred_raw_monthly.csv
Metadata: /Users/Sumaitat/Documents/Coding/ML_LaborTightnessRegimeShift/data/raw/fred/fred_metadata.csv
Failed Series: /Users/Sumaitat/Documents/Coding/ML_LaborTightnessRegimeShift/data/raw/fred/fred_failed_series.csv
Missing Summary: /Users/Sumaitat/Documents/Coding/ML_LaborTightnessRegimeShift/data/raw/fred/fred_missing_summary.csv


,job_openings_level,hires_rate,hires_level,quits_rate,quits_level,total_separations_rate,total_separations_level,layoffs_discharges_level,layoffs_discharges_rate,unemployment_rate,unemployment_level,payrolls_total_nonfarm,payrolls_total_private,payrolls_manufacturing,payrolls_service_providing,payrolls_construction,prime_age_lfpr,labor_force_participation_rate,employment_population_ratio,u6_unemployment_rate,prime_age_unemployment_rate,average_weeks_unemployed,avg_hourly_earnings_total_private,avg_weekly_earnings_total_private,avg_weekly_hours_total_private,avg_weekly_earnings_manufacturing,avg_hourly_earnings_manufacturing,cpi_all_items,cpi_core,cpi_all_items_less_shelter,cpi_services_less_rent_of_shelter,cpi_services_less_energy_services,cpi_shelter,pce_price_index,pce_core_price_index,personal_saving_rate,real_disposable_personal_income,real_personal_income,real_personal_income_less_transfers,retail_sales_advance,real_retail_sales,industrial_production,industrial_production_manufacturing,capacity_utilization,housing_starts,building_permits,new_one_family_houses_sold,consumer_sentiment,durable_goods_orders,capital_goods_orders_ex_aircraft,commercial_industrial_loans,total_bank_credit,m2_money_stock,treasury_1m,treasury_3m,treasury_2y,treasury_10y,fed_funds_rate,aaa_corporate_yield,baa_corporate_yield,high_yield_oas,bbb_oas,breakeven_5y,forward_5y5y_inflation,michigan_inflation_1y,recession_indicator
date,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
2000-01-01,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,4.0,5708.0,131011.0,110440.0,17284.0,106382.0,6752.0,84.4,67.3,64.6,7.0,12.7,13.1,NaN,NaN,NaN,14.12,NaN,169.3,179.3,162.8,198.6,199.2,180.9,72.961,74.306,324.2,9799.9,11441.669,10000.3,261545.0,154486.0,91.5380,92.2097,80.8292,1636.0,1727.0,873.0,112.0,201360.0,63975.0,1003.1482,4535.67470,4667.6,NaN,5.499000,6.440000,6.66,5.45,7.78,8.33,4.784286,1.548571,NaN,NaN,3.0,0.0
2000-02-01,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,4.1,5858.0,131120.0,110521.0,17284.0,106512.0,6730.0,84.4,67.3,64.6,7.1,10.8,12.6,NaN,NaN,NaN,14.14,NaN,170.0,179.4,163.6,199.2,199.5,181.3,73.191,74.415,289.4,9837.9,11488.814,10045.9,265686.0,156286.0,91.8239,92.4168,80.6792,1737.0,1692.0,856.0,111.3,183911.0,58497.0,1016.2395,4554.79750,4680.9,NaN,5.727000,6.610500,6.52,5.73,7.68,8.29,4.902381,1.547143,NaN,NaN,2.9,0.0
2000-03-01,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,4.0,5733.0,131604.0,110871.0,17302.0,106898.0,6811.0,84.3,67.3,64.6,7.1,11.0,12.7,NaN,NaN,NaN,14.17,NaN,171.0,180.0,164.7,199.9,200.2,181.9,73.505,74.568,276.5,9864.0,11520.777,10078.6,269019.0,157321.0,92.1504,93.0135,80.8587,1604.0,1651.0,900.0,107.1,192130.0,63642.0,1026.2416,4609.44284,4711.7,NaN,5.863913,6.528261,6.26,5.85,7.68,8.37,5.368696,1.720870,NaN,NaN,3.2,0.0
2000-04-01,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,3.8,5481.0,131884.0,111082.0,17299.0,107195.0,6794.0,84.4,67.3,64.7,6.9,10.7,12.4,NaN,NaN,NaN,14.23,NaN,170.9,180.3,164.5,200.2,200.6,182.3,73.444,74.617,311.7,9913.7,11582.719,10127.6,264067.0,154516.0,92.6989,93.5958,81.0265,1626.0,1597.0,841.0,109.2,195044.0,64414.0,1035.2058,4653.21385,4767.8,NaN,5.821579,6.403684,5.99,6.02,7.64,8.40,5.846500,2.041000,NaN,NaN,3.2,0.0
2000-05-01,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,4.0,5758.0,132105.0,110958.0,17278.0,107459.0,6770.0,84.1,67.1,64.4,7.1,11.1,12.6,NaN,NaN,NaN,14.22,NaN,171.2,180.7,164.7,200.9,201.2,182.8,73.505,74.697,314.7,9954.5,11624.340,10135.7,265992.0,155369.0,92.9499,93.6194,80.7010,1575.0,1543.0,857.0,110.7,188606.0,63753.0,1051.7891,4703.24964,4755.7,NaN,5.994545,6.809545,6.44,6.27,7.99,8.90,5.897826,2.179565,NaN,NaN,3.0,0.0


,column_name,series_id,title,frequency_short,units,seasonal_adjustment,last_updated,observation_start,observation_end,notes,raw_nonmissing_count,monthly_nonmissing_count,monthly_structural_missing_count,monthly_active_missing_count
0,aaa_corporate_yield,AAA,Moody's Seasoned Aaa Corporate Bond Yield,M,Percent,Not Seasonally Adjusted,2026-03-02 10:17:47-06,1919-01-01,2026-02-01,These instruments are based on bonds with matu...,314,314,0,0
1,average_weeks_unemployed,UEMPMEAN,Average Weeks Unemployed,M,Weeks,Seasonally Adjusted,2026-03-06 08:12:56-06,1948-01-01,2026-02-01,The series comes from the 'Current Population ...,313,313,0,1
2,avg_hourly_earnings_manufacturing,CES3000000003,"Average Hourly Earnings of All Employees, Manu...",M,Dollars per Hour,Seasonally Adjusted,2026-03-06 08:16:26-06,2006-03-01,2026-02-01,The series comes from the 'Current Employment ...,240,240,0,0
3,avg_hourly_earnings_total_private,CES0500000003,"Average Hourly Earnings of All Employees, Tota...",M,Dollars per Hour,Seasonally Adjusted,2026-03-06 08:16:10-06,2006-03-01,2026-02-01,The series comes from the 'Current Employment ...,240,240,0,0
4,avg_weekly_earnings_manufacturing,CES3000000008,Average Hourly Earnings of Production and Nons...,M,Dollars per Hour,Seasonally Adjusted,2026-03-06 08:16:29-06,1939-01-01,2026-02-01,Production and related employees include worki...,314,314,0,0
5,avg_weekly_earnings_total_private,CES0500000011,"Average Weekly Earnings of All Employees, Tota...",M,Dollars per Week,Seasonally Adjusted,2026-03-06 08:16:08-06,2006-03-01,2026-02-01,The series comes from the 'Current Employment ...,240,240,0,0
6,avg_weekly_hours_total_private,AWHAETP,"Average Weekly Hours of All Employees, Total P...",M,Hours,Seasonally Adjusted,2026-03-06 08:16:16-06,2006-03-01,2026-02-01,Average weekly hours relate to the average hou...,240,240,0,0
7,baa_corporate_yield,BAA,Moody's Seasoned Baa Corporate Bond Yield,M,Percent,Not Seasonally Adjusted,2026-03-02 10:17:45-06,1919-01-01,2026-02-01,These instruments are based on bonds with matu...,314,314,0,0
8,bbb_oas,BAMLC0A4CBBB,ICE BofA BBB US Corporate Index Option-Adjuste...,D,Percent,Not Seasonally Adjusted,2026-03-18 08:22:49-05,1996-12-31,2026-03-17,This data represents the Option-Adjusted Sprea...,6843,315,0,0
9,breakeven_5y,T5YIE,5-Year Breakeven Inflation Rate,D,Percent,Not Seasonally Adjusted,2026-03-18 16:02:43-05,2003-01-02,2026-03-18,The breakeven inflation rate represents a meas...,5805,279,0,0


,column_name,series_id,error
0,michigan_inflation_5y,MICH5Y,Bad Request. The series does not exist.


,missing_count,missing_pct,observation_start,observation_end,monthly_structural_missing_count,monthly_active_missing_count
column_name,,,,,,
avg_hourly_earnings_manufacturing,75,0.238095,2006-03-01,2026-02-01,0,0
avg_weekly_hours_total_private,75,0.238095,2006-03-01,2026-02-01,0,0
avg_weekly_earnings_total_private,75,0.238095,2006-03-01,2026-02-01,0,0
avg_hourly_earnings_total_private,75,0.238095,2006-03-01,2026-02-01,0,0
forward_5y5y_inflation,36,0.114286,2003-01-02,2026-03-18,0,0
breakeven_5y,36,0.114286,2003-01-02,2026-03-18,0,0
treasury_1m,18,0.057143,2001-07-31,2026-03-17,0,0
job_openings_level,13,0.041270,2000-12-01,2026-01-01,0,0
hires_rate,13,0.041270,2000-12-01,2026-01-01,0,0


In [21]:
external_urls = {
    "atlanta_fed_wage_growth_tracker": "https://www.atlantafed.org/-/media/Project/Atlanta/FRBA/Documents/datafiles/chcs/wage-growth-tracker/wage-growth-data.xlsx",
    "philly_fed_ruc_vintages": "https://www.philadelphiafed.org/-/media/FRBP/Assets/Surveys-And-Data/real-time-data/data-files/xlsx/rucQvMd.xlsx",
    "philly_fed_cpi_vintages": "https://www.philadelphiafed.org/-/media/FRBP/Assets/Surveys-And-Data/real-time-data/data-files/xlsx/cpiQvMd.xlsx",
    "philly_fed_spf_inflation": "https://www.philadelphiafed.org/-/media/FRBP/Assets/Surveys-And-Data/survey-of-professional-forecasters/historical-data/Inflation.xlsx",
}

In [22]:
downloaded_files = {}
external_download_rows = []
session = requests.Session()

for name, url in external_urls.items():
    out_path = external_dir / f"{name}.xlsx"

    try:
        response = request_with_retry(url, session=session, max_tries=5, timeout=120, sleep_seconds=2.0)

        with open(out_path, "wb") as f:
            f.write(response.content)

        downloaded_files[name] = out_path

        external_download_rows.append({
            "file_name": name,
            "url": url,
            "path": str(out_path),
            "status": "downloaded",
            "bytes": out_path.stat().st_size,
        })

    except Exception as e:
        external_download_rows.append({
            "file_name": name,
            "url": url,
            "path": str(out_path),
            "status": "failed",
            "bytes": np.nan,
            "error": str(e),
        })

external_download_log = pd.DataFrame(external_download_rows)
external_download_log_path = external_dir / "external_download_log.csv"
external_download_log.to_csv(external_download_log_path, index=False)

failed_external = external_download_log.loc[external_download_log["status"] != "downloaded"].copy()
if not failed_external.empty:
    raise RuntimeError(f"External downloads failed. See: {external_download_log_path}")

print("Downloaded External Files:")
for name, path in downloaded_files.items():
    print(name, "->", path)
print("External Download Log:", external_download_log_path)

Downloaded External Files:
atlanta_fed_wage_growth_tracker -> /Users/Sumaitat/Documents/Coding/ML_LaborTightnessRegimeShift/data/raw/external_sources/atlanta_fed_wage_growth_tracker.xlsx
philly_fed_ruc_vintages -> /Users/Sumaitat/Documents/Coding/ML_LaborTightnessRegimeShift/data/raw/external_sources/philly_fed_ruc_vintages.xlsx
philly_fed_cpi_vintages -> /Users/Sumaitat/Documents/Coding/ML_LaborTightnessRegimeShift/data/raw/external_sources/philly_fed_cpi_vintages.xlsx
philly_fed_spf_inflation -> /Users/Sumaitat/Documents/Coding/ML_LaborTightnessRegimeShift/data/raw/external_sources/philly_fed_spf_inflation.xlsx
External Download Log: /Users/Sumaitat/Documents/Coding/ML_LaborTightnessRegimeShift/data/raw/external_sources/external_download_log.csv


In [23]:
key_raw_dir = external_dir / "key_raw_tables"
key_raw_dir.mkdir(parents=True, exist_ok=True)

In [37]:
atl_path = downloaded_files["atlanta_fed_wage_growth_tracker"]
ruc_path = downloaded_files["philly_fed_ruc_vintages"]
cpi_path = downloaded_files["philly_fed_cpi_vintages"]
spf_path = downloaded_files["philly_fed_spf_inflation"]

atl_overall = pd.read_excel(atl_path, sheet_name="data_overall", engine="openpyxl")
atl_overall_12ma = pd.read_excel(atl_path, sheet_name="Overall 12ma", engine="openpyxl")

phl_ruc = pd.read_excel(ruc_path, sheet_name="ruc", engine="openpyxl")
phl_cpi = pd.read_excel(cpi_path, sheet_name="cpi", engine="openpyxl")
spf_inflation = pd.read_excel(spf_path, sheet_name="INFLATION", engine="openpyxl")

print("Raw Shapes")
print(". . .")
print("atl_overall:", atl_overall.shape)
print("atl_overall_12ma:", atl_overall_12ma.shape)
print("phl_ruc:", phl_ruc.shape)
print("phl_cpi:", phl_cpi.shape)
print("spf_inflation:", spf_inflation.shape)

display(atl_overall.head())
display(atl_overall_12ma.head())

display(phl_ruc.head())
display(phl_cpi.head())

display(spf_inflation.head())

Raw Shapes
. . .
atl_overall: (351, 17)
atl_overall_12ma: (352, 4)
phl_ruc: (949, 243)
phl_cpi: (949, 243)
spf_inflation: (225, 5)


/Users/Sumaitat/Documents/Coding/ML_LaborTightnessRegimeShift/.venv/lib/python3.14/site-packages/openpyxl/worksheet/header_footer.py:48: UserWarning: Cannot parse header or footer so it will be ignored
  warn("""Cannot parse header or footer so it will be ignored""")


,"Sources: Current Population Survey, Bureau of Labor Statistics, and Federal Reserve Bank of Atlanta Calculations. \nData updates can be found at https://www.frbatlanta/chcs/wage-growth-tracker. \nSee the 'definitions' tab in this spreadsheet for explanations of series. Wage computed on an hourly basis unless otherwise noted. Reported as 3-month moving averages of monthly series",Unnamed: 1,Unnamed: 2,Unnamed: 3,Unnamed: 4,Unnamed: 5,Unnamed: 6,Unnamed: 7,Unnamed: 8,Unnamed: 9,Unnamed: 10,Unnamed: 11,2026-01-12 00:00:00,Unnamed: 13,Unnamed: 14,Unnamed: 15,Unnamed: 16
0,NaT,Overall,Services,Full-time,College degree,Age 25-54,Female,Male,Job Stayer,Job Switcher,Paid Hourly,Overall: Weighted,Overall: Weighted 97,Overall: Weekly Basis,Overall: 25/20 trimmed mean,Lower 1/2 of wage distn,Upper 1/2 of wage distn
1,1997-01-01,.,.,.,.,.,.,.,.,.,.,.,.,.,.,.,.
2,1997-02-01,.,.,.,.,.,.,.,.,.,.,.,.,.,.,.,.
3,1997-03-01,4.5,4.6,4.5,4.7,4.4,4.6,4.4,4.1,5.2,4.2,4.9,4.9,4.8,4.5,4.8,4.2
4,1997-04-01,4.6,4.7,4.6,4.6,4.5,4.5,4.6,4.1,5.4,4.3,5,5,4.9,4.5,4.9,4.2


,"Sources: Current Population Survey, Bureau of Labor Statistics, and Federal Reserve Bank of Atlanta Calculations. \nData updates can be found at https://www.frbatlanta/chcs/wage-growth-tracker.",Unnamed: 1,Unnamed: 2,Unnamed: 3
0,The data are 12 month moving averages of month...,NaN,NaN,NaN
1,NaN,Overall,Overall: Weighted,Overall: Weighted 97
2,1997-01-01 00:00:00,.,.,.
3,1997-02-01 00:00:00,.,.,.
4,1997-03-01 00:00:00,.,.,.


,DATE,RUC65Q4,RUC66Q1,RUC66Q2,RUC66Q3,RUC66Q4,RUC67Q1,RUC67Q2,RUC67Q3,RUC67Q4,RUC68Q1,RUC68Q2,RUC68Q3,RUC68Q4,RUC69Q1,RUC69Q2,RUC69Q3,RUC69Q4,RUC70Q1,RUC70Q2,RUC70Q3,RUC70Q4,RUC71Q1,RUC71Q2,RUC71Q3,RUC71Q4,RUC72Q1,RUC72Q2,RUC72Q3,RUC72Q4,RUC73Q1,RUC73Q2,RUC73Q3,RUC73Q4,RUC74Q1,RUC74Q2,RUC74Q3,RUC74Q4,RUC75Q1,RUC75Q2,RUC75Q3,RUC75Q4,RUC76Q1,RUC76Q2,RUC76Q3,RUC76Q4,RUC77Q1,RUC77Q2,RUC77Q3,RUC77Q4,...,RUC13Q4,RUC14Q1,RUC14Q2,RUC14Q3,RUC14Q4,RUC15Q1,RUC15Q2,RUC15Q3,RUC15Q4,RUC16Q1,RUC16Q2,RUC16Q3,RUC16Q4,RUC17Q1,RUC17Q2,RUC17Q3,RUC17Q4,RUC18Q1,RUC18Q2,RUC18Q3,RUC18Q4,RUC19Q1,RUC19Q2,RUC19Q3,RUC19Q4,RUC20Q1,RUC20Q2,RUC20Q3,RUC20Q4,RUC21Q1,RUC21Q2,RUC21Q3,RUC21Q4,RUC22Q1,RUC22Q2,RUC22Q3,RUC22Q4,RUC23Q1,RUC23Q2,RUC23Q3,RUC23Q4,RUC24Q1,RUC24Q2,RUC24Q3,RUC24Q4,RUC25Q1,RUC25Q2,RUC25Q3,RUC25Q4,RUC26Q1
0,1947:01,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,1947:02,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,1947:03,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,1947:04,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,1947:05,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


,DATE,CPI65Q4,CPI66Q1,CPI66Q2,CPI66Q3,CPI66Q4,CPI67Q1,CPI67Q2,CPI67Q3,CPI67Q4,CPI68Q1,CPI68Q2,CPI68Q3,CPI68Q4,CPI69Q1,CPI69Q2,CPI69Q3,CPI69Q4,CPI70Q1,CPI70Q2,CPI70Q3,CPI70Q4,CPI71Q1,CPI71Q2,CPI71Q3,CPI71Q4,CPI72Q1,CPI72Q2,CPI72Q3,CPI72Q4,CPI73Q1,CPI73Q2,CPI73Q3,CPI73Q4,CPI74Q1,CPI74Q2,CPI74Q3,CPI74Q4,CPI75Q1,CPI75Q2,CPI75Q3,CPI75Q4,CPI76Q1,CPI76Q2,CPI76Q3,CPI76Q4,CPI77Q1,CPI77Q2,CPI77Q3,CPI77Q4,...,CPI13Q4,CPI14Q1,CPI14Q2,CPI14Q3,CPI14Q4,CPI15Q1,CPI15Q2,CPI15Q3,CPI15Q4,CPI16Q1,CPI16Q2,CPI16Q3,CPI16Q4,CPI17Q1,CPI17Q2,CPI17Q3,CPI17Q4,CPI18Q1,CPI18Q2,CPI18Q3,CPI18Q4,CPI19Q1,CPI19Q2,CPI19Q3,CPI19Q4,CPI20Q1,CPI20Q2,CPI20Q3,CPI20Q4,CPI21Q1,CPI21Q2,CPI21Q3,CPI21Q4,CPI22Q1,CPI22Q2,CPI22Q3,CPI22Q4,CPI23Q1,CPI23Q2,CPI23Q3,CPI23Q4,CPI24Q1,CPI24Q2,CPI24Q3,CPI24Q4,CPI25Q1,CPI25Q2,CPI25Q3,CPI25Q4,CPI26Q1
0,1947:01,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,21.48,21.48,21.48,21.48,21.48,21.48,21.48,21.48,21.48,21.48,21.48,21.48,21.48,21.48,21.48,21.48,21.48,21.48,21.48,21.48,21.48,21.48,21.48,21.48,21.48,21.48,21.48,21.48,21.48,21.48,21.48,21.48,21.48,21.48,21.48,21.48,21.48,21.48,21.48,21.48,21.48,21.48,21.48,21.48,21.48,21.48,21.48,21.48,21.48,21.48
1,1947:02,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,21.62,21.62,21.62,21.62,21.62,21.62,21.62,21.62,21.62,21.62,21.62,21.62,21.62,21.62,21.62,21.62,21.62,21.62,21.62,21.62,21.62,21.62,21.62,21.62,21.62,21.62,21.62,21.62,21.62,21.62,21.62,21.62,21.62,21.62,21.62,21.62,21.62,21.62,21.62,21.62,21.62,21.62,21.62,21.62,21.62,21.62,21.62,21.62,21.62,21.62
2,1947:03,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,22.00,22.00,22.00,22.00,22.00,22.00,22.00,22.00,22.00,22.00,22.00,22.00,22.00,22.00,22.00,22.00,22.00,22.00,22.00,22.00,22.00,22.00,22.00,22.00,22.00,22.00,22.00,22.00,22.00,22.00,22.00,22.00,22.00,22.00,22.00,22.00,22.00,22.00,22.00,22.00,22.00,22.00,22.00,22.00,22.00,22.00,22.00,22.00,22.00,22.00
3,1947:04,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,22.00,22.00,22.00,22.00,22.00,22.00,22.00,22.00,22.00,22.00,22.00,22.00,22.00,22.00,22.00,22.00,22.00,22.00,22.00,22.00,22.00,22.00,22.00,22.00,22.00,22.00,22.00,22.00,22.00,22.00,22.00,22.00,22.00,22.00,22.00,22.00,22.00,22.00,22.00,22.00,22.00,22.00,22.00,22.00,22.00,22.00,22.00,22.00,22.00,22.00
4,1947:05,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,21.95,21.95,21.95,21.95,21.95,21.95,21.95,21.95,21.95,21.95,21.95,21.95,21.95,21.95,21.95,21.95,21.95,21.95,21.95,21.95,21.95,21.95,21.95,21.95,21.95,21.95,21.95,21.95,21.95,21.95,21.95,21.95,21.95,21.95,21.95,21.95,21.95,21.95,21.95,21.95,21.95,21.95,21.95,21.95,21.95,21.95,21.95,21.95,21.95,21.95


,YEAR,QUARTER,INFPGDP1YR,INFCPI1YR,INFCPI10YR
0,1970,1,NaN,NaN,NaN
1,1970,2,2.9851,NaN,NaN
2,1970,3,3.7037,NaN,NaN
3,1970,4,3.5414,NaN,NaN
4,1971,1,3.5303,NaN,NaN


In [25]:
atl_overall_path = key_raw_dir / "atlanta_fed_wgt_overall_raw.csv"
atl_overall_12ma_path = key_raw_dir / "atlanta_fed_wgt_overall_12ma_raw.csv"
phl_ruc_path = key_raw_dir / "philly_fed_ruc_vintages_raw.csv"
phl_cpi_path = key_raw_dir / "philly_fed_cpi_vintages_raw.csv"
spf_inflation_path = key_raw_dir / "philly_fed_spf_inflation_raw.csv"

atl_overall.to_csv(atl_overall_path, index=False)
atl_overall_12ma.to_csv(atl_overall_12ma_path, index=False)
phl_ruc.to_csv(phl_ruc_path, index=False)
phl_cpi.to_csv(phl_cpi_path, index=False)
spf_inflation.to_csv(spf_inflation_path, index=False)

In [26]:
external_manifest = pd.DataFrame(
    [
        {
            "table_name": "atl_overall",
            "path": str(atl_overall_path),
            "rows": len(atl_overall),
            "cols": atl_overall.shape[1],
        },
        {
            "table_name": "atl_overall_12ma",
            "path": str(atl_overall_12ma_path),
            "rows": len(atl_overall_12ma),
            "cols": atl_overall_12ma.shape[1],
        },
        {
            "table_name": "phl_ruc",
            "path": str(phl_ruc_path),
            "rows": len(phl_ruc),
            "cols": phl_ruc.shape[1],
        },
        {
            "table_name": "phl_cpi",
            "path": str(phl_cpi_path),
            "rows": len(phl_cpi),
            "cols": phl_cpi.shape[1],
        },
        {
            "table_name": "spf_inflation",
            "path": str(spf_inflation_path),
            "rows": len(spf_inflation),
            "cols": spf_inflation.shape[1],
        },
    ]
)

In [36]:
external_manifest_path = external_dir / "external_key_table_manifest.csv"
external_manifest.to_csv(external_manifest_path, index=False)

print("Key Raw Directory:", key_raw_dir)
print("External Manifest:", external_manifest_path)

display(external_manifest)

Key Raw Directory: /Users/Sumaitat/Documents/Coding/ML_LaborTightnessRegimeShift/data/raw/external_sources/key_raw_tables
External Manifest: /Users/Sumaitat/Documents/Coding/ML_LaborTightnessRegimeShift/data/raw/external_sources/external_key_table_manifest.csv


,table_name,path,rows,cols
0,atl_overall,/Users/Sumaitat/Documents/Coding/ML_LaborTight...,351,17
1,atl_overall_12ma,/Users/Sumaitat/Documents/Coding/ML_LaborTight...,352,4
2,phl_ruc,/Users/Sumaitat/Documents/Coding/ML_LaborTight...,949,243
3,phl_cpi,/Users/Sumaitat/Documents/Coding/ML_LaborTight...,949,243
4,spf_inflation,/Users/Sumaitat/Documents/Coding/ML_LaborTight...,225,5


In [33]:
required_fred_cols = [
    "job_openings_level",
    "hires_rate",
    "quits_rate",
    "unemployment_rate",
    "cpi_all_items",
    "cpi_core",
    "pce_price_index",
    "pce_core_price_index",
    "fed_funds_rate",
]

missing_required_fred = [c for c in required_fred_cols if c not in fred_raw.columns]

In [34]:
summary_rows = [
    {"metric": "fred_rows", "value": len(fred_raw)},
    {"metric": "fred_cols", "value": fred_raw.shape[1]},
    {"metric": "fred_failed_series_count", "value": len(fred_failed)},
    {"metric": "external_downloaded_file_count", "value": int((external_download_log["status"] == "downloaded").sum())},
    {"metric": "external_failed_file_count", "value": int((external_download_log["status"] != "downloaded").sum())},
    {"metric": "missing_required_fred_count", "value": len(missing_required_fred)},
    {"metric": "fred_start", "value": str(fred_raw.index.min())},
    {"metric": "fred_end", "value": str(fred_raw.index.max())},
]


acquisition_summary = pd.DataFrame(summary_rows)
acquisition_summary_path = raw_dir / "acquisition_summary.csv"
acquisition_summary.to_csv(acquisition_summary_path, index=False)

In [35]:
print("Final Acquisition Summary")
print(". . .")
print(acquisition_summary)

if missing_required_fred:
    print("Missing Required FRED Columns:", missing_required_fred)

if not fred_failed.empty:
    print("Failed FRED Series Logged At:", fred_failed_path)

print("Acquisition Summary Path:", acquisition_summary_path)

Final Acquisition Summary
. . .
                           metric                value
0                       fred_rows                  315
1                       fred_cols                   66
2        fred_failed_series_count                    1
3  external_downloaded_file_count                    4
4      external_failed_file_count                    0
5     missing_required_fred_count                    0
6                      fred_start  2000-01-01 00:00:00
7                        fred_end  2026-03-01 00:00:00
Failed FRED Series Logged At: /Users/Sumaitat/Documents/Coding/ML_LaborTightnessRegimeShift/data/raw/fred/fred_failed_series.csv
Acquisition Summary Path: /Users/Sumaitat/Documents/Coding/ML_LaborTightnessRegimeShift/data/raw/acquisition_summary.csv
